# SVM & Vectorization Demo

This notebook walks through:
1. **What is Vectorization?** — Turning text into numbers a machine can understand
2. **How the same word gets different vector values** depending on context
3. **How SVM (Support Vector Machine) uses those vectors** to classify text
4. **Image Classification with SVM** — A visual example using pixel data

---
## Step 0: Install & Import Libraries
Run this cell first to make sure everything is available.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import load_digits
import matplotlib.pyplot as plt

print("All imports successful!")

---
## Part 1: Two Sentences, One Common Word

Let's take two simple sentences that share the word **"valve"** but mean very different things:

| Sentence | Intended Category |
|----------|------------------|
| `"valve assembly for compressor unit"` | Compressor Parts |
| `"valve stem seal kit for refrigerant line"` | Refrigerant Parts |

Both contain **"valve"**, but the surrounding words change how the machine sees them.

In [ ]:
# Our two sentences with the common word "valve"
sentence_1 = "valve assembly for compressor unit"
sentence_2 = "valve stem seal kit for refrigerant line"

sentences = [sentence_1, sentence_2]

print("Sentence 1:", sentence_1)
print("Sentence 2:", sentence_2)
print("\nCommon word: 'valve'")

---
## Part 2: CountVectorizer — Simple Word Counting

The simplest approach: count how many times each word appears.

Each sentence becomes a row of numbers — one number per unique word in the entire vocabulary.

In [ ]:
# Step 1: Fit a CountVectorizer on both sentences
count_vec = CountVectorizer()
count_matrix = count_vec.fit_transform(sentences)

# Show the vocabulary (each word gets an index)
vocab = count_vec.get_feature_names_out()
print("Vocabulary (all unique words):")
print(vocab)
print(f"\nTotal unique words: {len(vocab)}")

In [ ]:
# Step 2: Show the vector output for each sentence
count_df = pd.DataFrame(
    count_matrix.toarray(),
    columns=vocab,
    index=["Sentence 1", "Sentence 2"]
)

print("=" * 60)
print("COUNT VECTORIZER OUTPUT")
print("=" * 60)
print(count_df.to_string())
print("\n--- Key Observation ---")
print(f"'valve' column: Sentence 1 = {count_df.loc['Sentence 1', 'valve']}, Sentence 2 = {count_df.loc['Sentence 2', 'valve']}")
print("Both sentences have the SAME value for 'valve' (1).")
print("The difference comes from the OTHER words.")

---
## Part 3: TF-IDF Vectorizer — Weighted Word Importance

**TF-IDF** = Term Frequency × Inverse Document Frequency

- **TF**: How often a word appears in THIS sentence
- **IDF**: How rare the word is across ALL sentences

Words that appear in EVERY sentence (like "valve" and "for") get **lower scores**.  
Words unique to one sentence get **higher scores**.

This is exactly what our SVM production code uses (`TfidfVectorizer` + `SVC`).

In [ ]:
# Step 1: Fit a TF-IDF Vectorizer on both sentences
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(sentences)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray().round(4),
    columns=tfidf_vec.get_feature_names_out(),
    index=["Sentence 1", "Sentence 2"]
)

print("=" * 60)
print("TF-IDF VECTORIZER OUTPUT")
print("=" * 60)
print(tfidf_df.to_string())
print("\n--- Key Observation ---")
print(f"'valve' in Sentence 1: {tfidf_df.loc['Sentence 1', 'valve']:.4f}")
print(f"'valve' in Sentence 2: {tfidf_df.loc['Sentence 2', 'valve']:.4f}")
print(f"'for'   in Sentence 1: {tfidf_df.loc['Sentence 1', 'for']:.4f}")
print(f"'for'   in Sentence 2: {tfidf_df.loc['Sentence 2', 'for']:.4f}")
print("\nShared words ('valve', 'for') get LOWER TF-IDF scores because they appear in both sentences.")
print("Unique words ('compressor', 'refrigerant', etc.) get HIGHER scores — they are more informative.")

---
## Part 4: Side-by-Side Comparison

Let's visually compare how the two vectorizers treat the **same sentences**.

In [ ]:
# Side-by-side comparison for Sentence 1
compare_s1 = pd.DataFrame({
    "Word": vocab,
    "CountVec (Sentence 1)": count_matrix.toarray()[0],
    "TF-IDF (Sentence 1)": tfidf_matrix.toarray()[0].round(4)
})

compare_s2 = pd.DataFrame({
    "Word": vocab,
    "CountVec (Sentence 2)": count_matrix.toarray()[1],
    "TF-IDF (Sentence 2)": tfidf_matrix.toarray()[1].round(4)
})

print("=" * 50)
print("SENTENCE 1: 'valve assembly for compressor unit'")
print("=" * 50)
print(compare_s1.to_string(index=False))

print("\n" + "=" * 50)
print("SENTENCE 2: 'valve stem seal kit for refrigerant line'")
print("=" * 50)
print(compare_s2.to_string(index=False))

print("\n--- Summary ---")
print("CountVec: All present words = 1, absent words = 0 (no weighting)")
print("TF-IDF:   Shared words get LOWER scores, unique words get HIGHER scores")
print("\nThis is WHY TF-IDF helps SVM distinguish between similar descriptions!")

In [ ]:
# Visual bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CountVectorizer
x = np.arange(len(vocab))
width = 0.35
axes[0].bar(x - width/2, count_matrix.toarray()[0], width, label='Sentence 1', color='steelblue')
axes[0].bar(x + width/2, count_matrix.toarray()[1], width, label='Sentence 2', color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(vocab, rotation=45, ha='right')
axes[0].set_title('CountVectorizer Output')
axes[0].set_ylabel('Count')
axes[0].legend()

# TF-IDF
axes[1].bar(x - width/2, tfidf_matrix.toarray()[0], width, label='Sentence 1', color='steelblue')
axes[1].bar(x + width/2, tfidf_matrix.toarray()[1], width, label='Sentence 2', color='coral')
axes[1].set_xticks(x)
axes[1].set_xticklabels(vocab, rotation=45, ha='right')
axes[1].set_title('TF-IDF Vectorizer Output')
axes[1].set_ylabel('TF-IDF Score')
axes[1].legend()

plt.suptitle('Same Sentences, Different Vectorization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Notice: In TF-IDF, shared words ('valve', 'for') have EQUAL but REDUCED scores.")
print("The unique words stand out more — this helps SVM draw a better decision boundary.")

---
## Part 5: Watch the Vector Change When We Add More Documents

TF-IDF scores are **relative to the entire corpus**. Adding more sentences changes the vectors.

Let's see what happens when we add a third sentence.

In [ ]:
# Original: 2 sentences
corpus_small = [
    "valve assembly for compressor unit",
    "valve stem seal kit for refrigerant line"
]

# Expanded: add a third sentence that also mentions "valve"
corpus_expanded = [
    "valve assembly for compressor unit",
    "valve stem seal kit for refrigerant line",
    "valve pressure regulator for cooling system"
]

# Vectorize both
tfidf_small = TfidfVectorizer()
tfidf_expanded = TfidfVectorizer()

matrix_small = tfidf_small.fit_transform(corpus_small)
matrix_expanded = tfidf_expanded.fit_transform(corpus_expanded)

# Show "valve" score for Sentence 1 in both cases
valve_idx_small = list(tfidf_small.get_feature_names_out()).index("valve")
valve_idx_expanded = list(tfidf_expanded.get_feature_names_out()).index("valve")

print("=" * 60)
print("HOW 'valve' SCORE CHANGES FOR SENTENCE 1")
print("=" * 60)
print(f"With 2 sentences: valve = {matrix_small.toarray()[0][valve_idx_small]:.4f}")
print(f"With 3 sentences: valve = {matrix_expanded.toarray()[0][valve_idx_expanded]:.4f}")
print("\nThe score DECREASED because 'valve' now appears in ALL 3 sentences.")
print("TF-IDF automatically reduces the weight of common words.")
print("\nThis is why our production model retrains when new reference data is added —")
print("the vectors shift as the corpus grows.")

---
## Part 6: SVM in Action — Text Classification

Now let's see the full pipeline: **TF-IDF + SVM** classifying product descriptions.

This mirrors what our production code does in Block-2.

In [ ]:
# Sample training data (product descriptions and their categories)
train_descriptions = [
    "compressor assembly valve kit",
    "compressor motor bearing replacement",
    "compressor discharge valve plate",
    "compressor scroll set high pressure",
    "refrigerant valve service port adapter",
    "refrigerant recovery unit hose set",
    "refrigerant charging scale digital",
    "refrigerant leak detector sensor probe",
    "filter drier core replacement cartridge",
    "filter housing gasket seal kit",
    "filter suction line strainer mesh",
    "filter air intake panel washable"
]

train_labels = [
    "Compressor", "Compressor", "Compressor", "Compressor",
    "Refrigerant", "Refrigerant", "Refrigerant", "Refrigerant",
    "Filtration", "Filtration", "Filtration", "Filtration"
]

print("Training Data:")
for desc, label in zip(train_descriptions, train_labels):
    print(f"  [{label:12s}] {desc}")

In [ ]:
# Build the pipeline (same as production: TfidfVectorizer + SVC)
model = make_pipeline(TfidfVectorizer(), SVC(kernel='linear', probability=True))

# Train the model
model.fit(train_descriptions, train_labels)
print("Model trained successfully!")
print(f"Pipeline steps: {[step[0] for step in model.steps]}")

In [ ]:
# Test with new descriptions the model has NEVER seen
test_descriptions = [
    "valve assembly for compressor unit",          # Has 'valve' + 'compressor'
    "valve stem seal kit for refrigerant line",    # Has 'valve' + 'refrigerant'
    "filter replacement cartridge drier",           # filter + drier
    "compressor oil separator element",             # compressor context
    "refrigerant pressure gauge manifold set"       # refrigerant context
]

print("=" * 70)
print("PREDICTIONS ON NEW DESCRIPTIONS")
print("=" * 70)

for desc in test_descriptions:
    prediction = model.predict([desc])[0]
    probabilities = model.predict_proba([desc])[0]
    confidence = max(probabilities) * 100
    
    print(f"\nInput:       '{desc}'")
    print(f"Predicted:   {prediction}")
    print(f"Confidence:  {confidence:.1f}%")
    print(f"All scores:  {dict(zip(model.classes_, [f'{p:.1%}' for p in probabilities]))}")

print("\n" + "=" * 70)
print("Notice: Both sentences contain 'valve', but SVM correctly classifies")
print("them into DIFFERENT categories based on the surrounding context words.")
print("This is the power of TF-IDF + SVM working together.")

---
## Part 7: SVM with Images — Handwritten Digit Recognition

SVM isn't just for text. It works on **any numerical data**, including images.

Here we use the classic **digits dataset** — 8x8 pixel images of handwritten digits (0-9).  
Each image is "vectorized" as 64 pixel values (8 rows × 8 columns).

| Text Vectorization | Image Vectorization |
|-------------------|--------------------|
| Words → TF-IDF scores | Pixels → Grayscale values (0-16) |
| Each word = one dimension | Each pixel = one dimension |
| 5-word sentence → 5+ dimensional vector | 8×8 image → 64-dimensional vector |

In [ ]:
# Load the digits dataset (built into scikit-learn)
digits = load_digits()

print(f"Dataset: {digits.data.shape[0]} images")
print(f"Each image: {digits.images[0].shape[0]}x{digits.images[0].shape[1]} pixels = {digits.data.shape[1]} features")
print(f"Classes: digits {list(range(10))}")

# Show some sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray_r', interpolation='nearest')
    ax.set_title(f"Label: {digits.target[i]}", fontsize=12)
    ax.axis('off')

plt.suptitle('Sample Handwritten Digits (8x8 pixels)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Show how an image becomes a vector
sample_image = digits.images[0]
sample_vector = digits.data[0]

print("=" * 50)
print("IMAGE TO VECTOR CONVERSION")
print("=" * 50)
print(f"\nOriginal 8x8 image (pixel grid):")
print(sample_image.astype(int))
print(f"\nFlattened to 64-element vector:")
print(sample_vector.astype(int))
print(f"\nThis digit is: {digits.target[0]}")
print("\nJust like TF-IDF turns words into numbers,")
print("image vectorization turns pixels into numbers.")
print("SVM finds the best boundary between these number vectors.")

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=42
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

# Train SVM on image data
image_svm = SVC(kernel='linear', probability=True)
image_svm.fit(X_train, y_train)

# Evaluate
y_pred = image_svm.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nAccuracy: {accuracy:.1%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Show predictions on test images
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i].reshape(8, 8), cmap='gray_r', interpolation='nearest')
    pred = y_pred[i]
    actual = y_test[i]
    color = 'green' if pred == actual else 'red'
    ax.set_title(f"Pred: {pred} | Actual: {actual}", fontsize=11, color=color)
    ax.axis('off')

plt.suptitle('SVM Predictions on Handwritten Digits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Green = Correct prediction")
print("Red   = Wrong prediction")

---
## Summary

| Concept | What It Does | Example |
|---------|-------------|--------|
| **CountVectorizer** | Counts word occurrences | `"valve" → 1` (same in both sentences) |
| **TF-IDF Vectorizer** | Weights words by rarity | `"valve" → 0.33` (lower because shared) |
| **SVM** | Finds the best dividing line between categories | Separates Compressor vs Refrigerant parts |
| **Image Vectorization** | Flattens pixel grid into numbers | 8×8 image → 64-number vector |

### Key Takeaways
1. **Same word, different vectors** — TF-IDF gives context-aware weights; shared words score lower
2. **Vectors change when corpus changes** — Adding more documents shifts all TF-IDF scores
3. **SVM works on any numerical vectors** — text, images, or any structured data
4. **Our production pipeline** uses `TfidfVectorizer() + SVC(kernel='linear')` to classify part descriptions into taxonomy nodes